# Día 5 - PCA, selección de variables e ingeniería de características

**Fecha:** 19-10-2026

Hoy respondemos a tres problemas distintos:

1. **PCA:** tengo muchas variables y quiero representar la mayor variación con menos dimensiones.
2. **Selección:** quiero decidir qué variables conservar.
3. **Feature engineering:** quiero construir representaciones más útiles a partir de datos existentes.

Reducir dimensionalidad no significa automáticamente mejorar predicción.

## 0. Librerías y carpeta de datos

No hay instalación automática. Si una librería opcional falta, el notebook mostrará el comando que puedes ejecutar manualmente en una celda nueva. Todos los imports principales quedan juntos para que sea claro qué herramientas usa la jornada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.pipeline import make_pipeline
from statsmodels.stats.outliers_influence import variance_inflation_factor

In [ ]:
DATA_DIR = Path('../datos')
if not DATA_DIR.exists():
    DATA_DIR = Path('datos')
print('Datos:', DATA_DIR.resolve())

A partir de aquí, cada bloque de código irá acompañado de una explicación de **qué entra, qué sale y qué debemos mirar**. Ejecutar una celda sin interpretar el resultado no cuenta como comprender el modelo.

# 1. Volvemos a mirar los datos antes de transformar

Usaremos el dataset económico sintético del Día 1. Primero revisamos objetivo, tipos y correlaciones.

Aunque hoy el tema sea PCA, **no transformamos datos sin entenderlos primero**. PCA no arregla nulos, unidades mal definidas ni variables que contienen leakage. Primero revisamos el mismo dataset económico y recordamos el significado de cada columna.

In [ ]:
df = pd.read_csv(DATA_DIR/'actividad_economica_demo.csv', parse_dates=['fecha']).drop_duplicates()

for c in ['inflacion_pct','indice_confianza']:
    df[c]=df[c].fillna(df[c].median())

TARGET='crecimiento_actividad_pct'
features_num=['inflacion_pct','desempleo_pct','exportaciones_mill','inversion_mill','precio_petroleo_usd','indice_confianza']

df[features_num+[TARGET]].describe().T

In [ ]:
df[features_num+[TARGET]].corr().round(2)

### Cómo interpretar la matriz de correlación

Cada celda va de -1 a 1 y mide **relación lineal** entre un par de variables. La diagonal vale 1 porque cada variable está perfectamente correlacionada consigo misma.

Correlaciones altas entre dos X pueden señalar **multicolinealidad**. Eso puede volver inestables coeficientes lineales, aunque el modelo todavía prediga razonablemente.

# 2. ¿Por qué PCA exige pensar en escala?

PCA busca direcciones que concentran varianza. Si una variable está medida en miles y otra en porcentajes, la primera puede dominar solo por la unidad.

Por eso normalmente estandarizamos antes de PCA.

PCA busca direcciones que capturen la mayor varianza. Si una variable tiene varianza enorme solo por estar medida en miles, dominará los componentes.

Por eso estandarizamos:

$$z=\frac{x-\mu}{\sigma}$$

Después del escalado, cada columna tiene aproximadamente media 0 y desviación 1.

In [ ]:
X = df[features_num]

scaler=StandardScaler()
X_scaled=scaler.fit_transform(X)

print('Desviaciones antes:')
print(X.std().round(2))
print()
print('Desviaciones después:')
print(pd.DataFrame(X_scaled,columns=features_num).std().round(2))

## 2.1 Covarianza: la intuición previa a PCA

Si dos variables tienden a subir juntas, su covarianza es positiva; si una sube cuando la otra baja, negativa.

PCA analiza la estructura de covarianza/correlación para encontrar direcciones de máxima variación.

La covarianza entre X e Y puede escribirse conceptualmente como:

$$Cov(X,Y)=\frac{1}{n-1}\sum_i(x_i-\bar x)(y_i-\bar y)$$

- positiva → tienden a moverse en la misma dirección;
- negativa → direcciones opuestas;
- cerca de 0 → poca covariación lineal.

PCA busca combinaciones lineales relacionadas con esta estructura de covarianza.

In [ ]:
# Matriz de covarianza después de estandarizar.
cov=np.cov(X_scaled,rowvar=False)
pd.DataFrame(cov,index=features_num,columns=features_num).round(2)

## 2.2 Eigenvalues/eigenvectors sin profundizar en álgebra

Matemáticamente PCA encuentra vectores propios de la matriz de covarianza. Los eigenvalues indican cuánta varianza corresponde a cada dirección.

No necesitamos hacer un curso de álgebra lineal para usar PCA, pero sí entender la idea:

- **eigenvector / componente:** dirección del espacio;
- **eigenvalue:** cuánta varianza existe en esa dirección.

PC1 es la dirección que captura más varianza posible; PC2 captura la mayor varianza restante siendo ortogonal a PC1, y así sucesivamente.

In [ ]:
eigenvalues,eigenvectors=np.linalg.eigh(cov)
orden=np.argsort(eigenvalues)[::-1]

print('Eigenvalues ordenados:')
print(np.round(eigenvalues[orden],3))
print('Proporciones aproximadas:')
print(np.round(eigenvalues[orden]/eigenvalues.sum(),3))

# 3. PCA: componentes como combinaciones de variables

Un componente principal no es una columna original. Es una combinación lineal:

\[
PC_1 = a_1X_1 + a_2X_2 + ... + a_pX_p
\]

PC1 es la dirección que captura mayor variación; PC2 captura la mayor variación restante y es ortogonal a PC1, etc.

Cada componente es una combinación de las variables originales:

$$PC_1=w_1x_1+w_2x_2+\dots+w_px_p$$

Los pesos $w_j$ son **loadings**. Los componentes no son variables originales, por lo que su interpretación requiere mirar esos pesos.

In [ ]:
pca=PCA()
Z=pca.fit_transform(X_scaled)

var_exp=pd.DataFrame({
    'componente':np.arange(1,len(features_num)+1),
    'varianza_explicada':pca.explained_variance_ratio_,
    'acumulada':np.cumsum(pca.explained_variance_ratio_)
})
var_exp.round(3)

In [ ]:
plt.figure(figsize=(8,4))
plt.bar(var_exp['componente'],var_exp['varianza_explicada'],label='Individual')
plt.plot(var_exp['componente'],var_exp['acumulada'],marker='o',label='Acumulada')
plt.axhline(.80,linestyle='--',linewidth=1,label='80%')
plt.xlabel('Componente')
plt.ylabel('Proporción de varianza')
plt.title('Varianza explicada por PCA')
plt.legend()
plt.show()

## 3.1 Loadings: qué representa cada componente

Los `components_` muestran cuánto pesa cada variable original dentro del componente. El signo puede invertirse sin cambiar el significado geométrico; importa la magnitud y el patrón relativo.

Un loading grande en valor absoluto significa que esa variable contribuye fuertemente al componente. El signo indica dirección dentro del componente, pero el signo global de un eigenvector puede invertirse sin cambiar la solución matemática.

In [ ]:
loadings=pd.DataFrame(
    pca.components_.T,
    index=features_num,
    columns=[f'PC{i}' for i in range(1,len(features_num)+1)]
)

loadings.round(3)

In [ ]:
# Visualizamos cada observación en los dos primeros componentes.
plt.figure(figsize=(8,5))
plt.scatter(Z[:,0],Z[:,1],alpha=.5)
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.title('Datos proyectados a 2 dimensiones')
plt.show()

## 3.2 Reconstrucción y pérdida de información

Si conservamos menos componentes, ya no podemos reconstruir exactamente las variables originales. El objetivo es aceptar cierta pérdida a cambio de una representación más compacta.

Reducir dimensionalidad significa aceptar pérdida de información. Si usamos solo 2 componentes, podemos reconstruir una aproximación de los datos originales, pero no recuperar exactamente todo lo descartado.

In [ ]:
pca2=PCA(n_components=2).fit(X_scaled)
Z2=pca2.transform(X_scaled)
X_recon=pca2.inverse_transform(Z2)

error_recon=np.mean((X_scaled-X_recon)**2)
print('Error medio cuadrático de reconstrucción con 2 componentes:',round(error_recon,3))

## 3.3 PCA no “selecciona” columnas

PCA crea **nuevas variables mezclando las originales**. Si necesitas conservar nombres y significado original de variables, selección de features puede ser más apropiada.

PCA **crea nuevas variables** combinando las originales. Si la necesidad es conservar nombres originales por regulación/interpretabilidad, quizá sea preferible selección de variables en lugar de PCA.

# 4. PCA para visualización vs PCA para predicción

Que dos componentes expliquen 80% de la varianza de X no significa que conserven 80% de la información útil para predecir y.

La única forma de decidir si ayuda al modelo es comparar en Validation.

PCA puede usarse con dos objetivos distintos:

1. **visualización/exploración:** proyectar muchas dimensiones a 2D/3D;
2. **preprocesamiento predictivo:** alimentar un modelo con menos componentes.

Que la proyección 2D “se vea bonita” no demuestra que mejore una predicción.

In [ ]:
# Split antes de ajustar scaler/PCA para evitar leakage.
train_val, test = train_test_split(df,test_size=.15,random_state=42)
train, val = train_test_split(train_val,test_size=.1765,random_state=42)

# Modelo sin PCA.
m_base=LinearRegression().fit(train[features_num],train[TARGET])
p_base=m_base.predict(val[features_num])

# PCA debe aprender scaler y componentes SOLO desde Train.
sc=StandardScaler().fit(train[features_num])
Xtr=sc.transform(train[features_num])
Xv=sc.transform(val[features_num])

pca3=PCA(n_components=3).fit(Xtr)
Xtr_p=pca3.transform(Xtr)
Xv_p=pca3.transform(Xv)

m_pca=LinearRegression().fit(Xtr_p,train[TARGET])
p_pca=m_pca.predict(Xv_p)

comparacion=pd.DataFrame({
    'Sin_PCA':{
        'MAE':mean_absolute_error(val[TARGET],p_base),
        'RMSE':np.sqrt(mean_squared_error(val[TARGET],p_base)),
        'R2':r2_score(val[TARGET],p_base)
    },
    'PCA_3':{
        'MAE':mean_absolute_error(val[TARGET],p_pca),
        'RMSE':np.sqrt(mean_squared_error(val[TARGET],p_pca)),
        'R2':r2_score(val[TARGET],p_pca)
    }
}).T
comparacion.round(3)

## 4.1 Fuga de información en PCA

Error típico:

```python
scaler.fit_transform(TODO_EL_DATASET)
pca.fit_transform(TODO_EL_DATASET)
# y recién después hago split
```

Eso permite que Validation/Test influyan en medias, desviaciones y componentes. El split debe ocurrir antes de `fit` de transformaciones aprendidas.

Si hacemos `PCA.fit_transform(X_completo)` antes del split, los componentes se calculan usando también Validation/Test. Eso es **data leakage**.

En predicción, `Scaler.fit()` y `PCA.fit()` deben aprender parámetros **solo en Train**. Un Pipeline es la forma más segura de garantizarlo.

# 5. Selección de variables: preguntas distintas

Podemos preguntar:

- ¿está correlacionada con y?
- ¿aporta información adicional cuando están otras variables?
- ¿mejora Validation al incluirla?
- ¿el modelo depende de ella para predecir?

No existe una única definición de “variable importante”.

Selección de variables responde “¿qué columnas originales quiero conservar?”. Hay varias preguntas distintas:

- relación bivariada → correlación;
- evidencia inferencial condicional → p-values;
- valor predictivo fuera de muestra → ablation/permutation;
- redundancia entre X → VIF/correlación;
- selección regularizada → Lasso.

Ninguna herramienta aislada define “la verdad” sobre importancia.

## 5.0 Correlación Pearson vs Spearman

Pearson detecta relación lineal; Spearman usa rangos y detecta relaciones monótonas aunque sean curvas.

- **Pearson** detecta especialmente relación lineal.
- **Spearman** trabaja con rangos y detecta relaciones monótonas aunque sean curvas.

Si Spearman es alto y Pearson bastante menor, puede existir una relación creciente/decreciente no perfectamente lineal.

In [ ]:
pearson=df[features_num+[TARGET]].corr(method='pearson')[TARGET].drop(TARGET)
spearman=df[features_num+[TARGET]].corr(method='spearman')[TARGET].drop(TARGET)

pd.DataFrame({'Pearson':pearson,'Spearman':spearman}).round(3)

## 5.1 Ablation: quitar una variable

Es una prueba muy clara: mantener todo igual y retirar una feature. Si Validation empeora de forma estable, esa feature aportaba información predictiva.

Ablation es una pregunta muy práctica: **¿qué ocurre con Validation si elimino esta variable manteniendo el resto del proceso igual?**

Si quitar una feature apenas cambia MAE/R², su aporte predictivo adicional puede ser pequeño. Si el error empeora mucho y de forma estable, aportaba información.

In [ ]:
filas=[]

# Error del modelo con todas las features.
mae_full=mean_absolute_error(val[TARGET],p_base)

for quitar in features_num:
    cols=[c for c in features_num if c!=quitar]
    m=LinearRegression().fit(train[cols],train[TARGET])
    pred=m.predict(val[cols])
    filas.append({
        'variable_quitada':quitar,
        'MAE_val':mean_absolute_error(val[TARGET],pred),
        'cambio_MAE':mean_absolute_error(val[TARGET],pred)-mae_full
    })

pd.DataFrame(filas).sort_values('cambio_MAE',ascending=False).round(3)

## 5.2 Permutation importance

Mezclamos una columna en Validation. Si la métrica empeora, esa variable contenía información útil para el modelo.

Permutation importance rompe intencionalmente una variable mezclando sus valores. Si el modelo pierde rendimiento, esa feature era útil para sus predicciones.

Ventaja: mide impacto sobre una métrica de validación. Limitación: variables muy correlacionadas pueden reemplazarse mutuamente y parecer menos importantes individualmente.

In [ ]:
perm=permutation_importance(
    m_base,
    val[features_num],
    val[TARGET],
    n_repeats=20,
    random_state=42,
    scoring='neg_mean_absolute_error'
)

pd.Series(perm.importances_mean,index=features_num).sort_values(ascending=False)

## 5.3 p-values con statsmodels: inferencia, no ranking predictivo

El p-value pregunta por compatibilidad con un coeficiente nulo bajo los supuestos del modelo. No mide causalidad ni necesariamente importancia predictiva.

Un p-value pequeño no significa “variable predictivamente más importante” ni “causa”. Responde una pregunta inferencial bajo un modelo y supuestos concretos. Por eso lo contrastamos con validación y conocimiento del dominio.

In [ ]:

X_ols=sm.add_constant(df[features_num])
ols=sm.OLS(df[TARGET],X_ols).fit()
ols.summary2().tables[1]

## 5.4 Lasso como selección automática

Lasso puede llevar coeficientes a cero. El resultado depende de `alpha` y de la escala.

In [ ]:

for alpha in [0.001,.01,.05,.1,.5,1.0]:
    m=make_pipeline(StandardScaler(),Lasso(alpha=alpha,max_iter=20000))
    m.fit(train[features_num],train[TARGET])
    coef=m.named_steps['lasso'].coef_
    print(f'alpha={alpha:<5} coeficientes no cero={np.sum(np.abs(coef)>1e-8)}')

# 6. Multicolinealidad y VIF

Cuando dos variables contienen casi la misma información, los coeficientes lineales pueden volverse inestables.

VIF es una herramienta clásica para diagnosticar cuánto puede explicarse una feature a partir de las demás.

VIF pregunta cuánto puede explicarse una X mediante las otras X:

$$VIF_j=\frac{1}{1-R_j^2}$$

VIF alto indica redundancia lineal y puede hacer inestables los coeficientes. No es una regla automática para borrar columnas: primero revisa el significado y objetivo del modelo.

In [ ]:

X_vif = StandardScaler().fit_transform(df[features_num])

vif=pd.DataFrame({
    'variable':features_num,
    'VIF':[variance_inflation_factor(X_vif,i) for i in range(X_vif.shape[1])]
})
vif.sort_values('VIF',ascending=False).round(2)

# 7. Ingeniería de características

Feature engineering no significa crear columnas al azar. Una feature debe surgir de:

- conocimiento del dominio;
- patrón visual;
- hipótesis razonable;
- necesidad de representar no linealidad/interacción;
- mejora estable en Validation.

Ejemplos:

- ratio exportaciones/inversión;
- interacción inflación * desempleo;
- términos cuadráticos;
- lags para series de tiempo.

Feature engineering crea representaciones que expresan mejor el fenómeno. Ejemplos:

- razones (`exportaciones / inversión`);
- cambios interanuales;
- logaritmos para escalas sesgadas;
- lags y medias móviles en series;
- interacciones entre variables;
- términos polinómicos.

Una feature nueva debe justificarse y validarse fuera de muestra; crear cientos de combinaciones aumenta riesgo de overfitting.

In [ ]:
df_feat=df.copy()

# Ratio: ejemplo didáctico de relación entre dos magnitudes.
df_feat['exportaciones_por_inversion']=df_feat['exportaciones_mill']/df_feat['inversion_mill']

# Interacción: permite que el efecto conjunto difiera de la suma independiente.
df_feat['inflacion_x_desempleo']=df_feat['inflacion_pct']*df_feat['desempleo_pct']

# No linealidad simple.
df_feat['inflacion_cuadrado']=df_feat['inflacion_pct']**2

df_feat[['exportaciones_por_inversion','inflacion_x_desempleo','inflacion_cuadrado']].head()

## 7.1 Regresión polinómica e interacciones

Una regresión polinómica sigue siendo lineal respecto a sus coeficientes, pero recibe features transformadas como `x²` o `x1*x2`.

Una regresión polinómica sigue siendo lineal respecto a sus **coeficientes**, aunque use features transformadas:

$$\hat y=b_0+b_1x+b_2x^2$$

Una interacción permite que el efecto asociado a una variable dependa de otra:

$$\hat y=b_0+b_1x_1+b_2x_2+b_3(x_1x_2)$$

El grado y las interacciones son decisiones que deben evaluarse con Validation/CV.

In [ ]:
# Ejemplo con dos variables para que el número de features siga siendo manejable.
cols_poly=['inflacion_pct','desempleo_pct']
poly=PolynomialFeatures(degree=2,include_bias=False)

X_poly=poly.fit_transform(train[cols_poly])
print(poly.get_feature_names_out(cols_poly))

In [ ]:
# Comparamos grado 1 vs grado 2 en Validation.
modelo_poly=LinearRegression().fit(X_poly,train[TARGET])
X_val_poly=poly.transform(val[cols_poly])
pred_poly=modelo_poly.predict(X_val_poly)

print('MAE polinomio grado 2:',round(mean_absolute_error(val[TARGET],pred_poly),3))

Crear todas las interacciones posibles sin criterio puede aumentar complejidad y overfitting. Una feature nueva debe justificarse con dominio, patrón visual o mejora estable de Validation.

# 8. Regularización: Ridge y Lasso como puente a optimización

En modelos lineales con muchas variables:

- Ridge (L2) reduce coeficientes grandes;
- Lasso (L1) puede llevar algunos a cero.

La penalización depende de la escala, por eso normalmente estandarizamos.

Ridge/Lasso ya se estudiaron en detalle el Día 2. Aquí se conectan con selección/optimización:

- Ridge estabiliza coeficientes correlacionados;
- Lasso puede llevar algunos a 0;
- ambos deben validarse fuera de muestra y suelen beneficiarse del escalado.

In [ ]:

for alpha in [0.01,0.1,1,10,100]:
    ridge=make_pipeline(StandardScaler(),Ridge(alpha=alpha))
    ridge.fit(train[features_num],train[TARGET])
    pred=ridge.predict(val[features_num])
    print(f'alpha={alpha:6} -> MAE={mean_absolute_error(val[TARGET],pred):.3f}')

# Taller U4 - preparación

1. Decide cuántos componentes PCA conservarías y por qué.
2. Compara un modelo con y sin PCA.
3. Realiza ablation de variables.
4. Crea al menos una feature con justificación económica.
5. Comprueba si mejora Validation.
6. Explica por qué una mejora en Train no es suficiente.

## Preguntas para cerrar

1. ¿Por qué PCA suele requerir escalado?
2. ¿Qué es varianza explicada?
3. ¿Qué diferencia hay entre componente y variable original?
4. ¿Por qué PCA puede producir leakage?
5. ¿Qué diferencia hay entre selección de variables y reducción dimensional?
6. ¿Qué pregunta responde permutation importance?
7. ¿Por qué p-value no equivale a importancia predictiva ni causalidad?
8. ¿Qué problema advierte VIF?
9. ¿Cuándo una interacción puede tener sentido?